In [1]:
import struct
import ctypes
import sys

In [2]:
class PyListObject(ctypes.Structure):
  _fields_ = (
    ("obj_refcnt", ctypes.c_long),
    ("ob_type", ctypes.c_void_p),
    ("ob_size", ctypes.c_long),
    ("ob_item", ctypes.POINTER(ctypes.c_void_p)),
    ("allocated", ctypes.c_long),
  )

In [ ]:
a = [0.3, 0.6, 0.1]

pylist_obj = PyListObject.from_address(id(a))

for idx, s in enumerate(a):
  addr = pylist_obj.ob_item[idx]
  s_mem = ctypes.string_at(addr, sys.getsizeof(s))

  # Decode as three little-endian 64-bit values: two integers and one double
  int1, int2, float1 = struct.unpack("<qqd", s_mem)

  print(f"addr", addr, idx, int1, int2, float1)

addr 140384895291728 0 4 8824224 0.3
addr 140384895290448 1 4 8824224 0.6
addr 140384895291856 2 4 8824224 0.1


Using `numpy.array` will not cause a crash when accessing the memory address
instead of using built-in type `list`

In [8]:
import numpy as np

# Create a contiguous block of memory (doubles)
arr = np.array([0.3, 0.6, 0.1], dtype=np.float64)

# Get the raw memory address
addr = arr.__array_interface__['data'][0]
item_size = arr.itemsize  # 8 bytes for float64

print(f"Base Address: {hex(addr)}")
print(f"Item Size: {item_size}")

# Access manually via memory view (safer than ctypes)
for i in range(len(arr)):
    offset = i * item_size
    # Interpret memory at offset as a float
    val = np.frombuffer(arr.tobytes()[offset:offset+item_size], dtype=np.float64)[0]
    print(f"Index {i} at {hex(addr + offset)}: {val}")

Base Address: 0x1b52bf90
Item Size: 8
Index 0 at 0x1b52bf90: 0.3
Index 1 at 0x1b52bf98: 0.6
Index 2 at 0x1b52bfa0: 0.1
